In [76]:
import pandas as pd
import networkx as nx
import igraph as ig
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import re
from math import ceil  # Import ceil from math module
import math
from matplotlib import cm
from matplotlib.colors import Normalize
import gseapy as gp
def plot_enrichr_pathway_analysis(gene_list, keywords=None, gene_set='GO_Biological_Process_2023', organism='mouse', output_path='pathway_analysis_plot.png'):
    # Run Enrichr analysis
    try:
        # Run Enrichr analysis
        enrichr_results = gp.enrichr(
            gene_list=gene_list,
            gene_sets=gene_set,
            organism=organism
        )
    except Exception as e:
        # Handle failure gracefully
        print(f"Enrichr analysis failed: {e}")
        print("Skipping pathway analysis for this gene list.")
        return
    
    # Extract the results DataFrame
    enrichr_df = enrichr_results.results

    # Filter the DataFrame by the specific gene set 'GO_Biological_Process_2023'
    enrichr_df = enrichr_df[enrichr_df['Gene_set'] == gene_set]

    # Define a threshold for significance
    pval_threshold = 0.05  # Adjusted p-value cutoff

    # Filter by adjusted p-value (or use 'P-value' if 'Adjusted P-value' is unavailable)
    enrichr_df = enrichr_df[enrichr_df['Adjusted P-value'] <= pval_threshold]


    # Print the filtered DataFrame
    print("Number of confident pathways (start) : " + str(len(enrichr_df)))

    # Step 1: Filter pathways related to programmed cell death and apoptosis
    #filtered_enrichr_df = enrichr_df[enrichr_df['Term'].str.contains('|'.join(keywords), case=False, na=False)]
    def keyword_match(term, keywords):
        return any(keyword.lower() in term.lower() for keyword in keywords)

    if keywords:
        filtered_enrichr_df = enrichr_df[enrichr_df['Term'].apply(keyword_match, keywords=keywords)]
    else:
        filtered_enrichr_df = enrichr_df

    # Print the filtered DataFrame
    nconfpath = len(filtered_enrichr_df)
    if nconfpath == 0:  # Check if the filtered DataFrame is empty
        print("No pathways match the specified keywords. Exiting analysis.")
        return
    
    print("Number of confident pathways (of interest): " + str(nconfpath))

    def clean_pathway_name(name):
        # Remove "GO:" followed by numbers and the optional parentheses around GO IDs
        name = re.sub(r'\(GO:\d+\)', '', name)  # Removes "(GO:101010101)" style entries
        return name

    # Apply cleaning to the pathway names in the DataFrame
    #filtered_enrichr_df['Term'] = filtered_enrichr_df['Term'].apply(clean_pathway_name)
    filtered_enrichr_df.loc[:, 'Term'] = filtered_enrichr_df['Term'].apply(clean_pathway_name)

    # Step 2: Proceed with the graph creation using the filtered DataFrame
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform(filtered_enrichr_df['Term'])

    # Step 3: Build Graph from Shared Genes
    G = nx.Graph()
    gene_counts = {}
    pathway_gene_map = {}

    for _, row in filtered_enrichr_df.iterrows():
        genes = set(row['Genes'].split(';'))  # Convert to set for intersection checks
        pathway_gene_map[row['Term']] = genes
        gene_counts[row['Term']] = len(genes)  # Count genes per term

    #filtered_enrichr_df['num_genes'] = filtered_enrichr_df['Term'].map(gene_counts)
    filtered_enrichr_df = filtered_enrichr_df.copy()
    filtered_enrichr_df.loc[:, 'num_genes'] = filtered_enrichr_df['Term'].map(gene_counts)

    # Filter pathways that have >= 3 genes, with fallback to >= 2 and >= 1
    #for threshold in [3, 2, 1]:
    for threshold in [3, 2]:
        idx = filtered_enrichr_df['num_genes'] >= threshold
        if idx.sum() > 3:
            print(f"Pathways with >= {threshold} genes found: {idx.sum()}")
            filtered_enrichr_df = filtered_enrichr_df[idx]
            break
    else:
        # If no pathways meet the minimum gene count, exit the function
        print("No pathways with at least 2 gene found. Skipping further analysis.")
        return

    filtered_enrichr_df = filtered_enrichr_df[idx]

    # Recalculate gene counts for the filtered pathways
    gene_counts = {term: gene_counts[term] for term in filtered_enrichr_df['Term']}

    # Add edges only for shared genes
    shared_gene_labels = {}
    non_shared_gene_labels = {}
    for pathway1, genes1 in pathway_gene_map.items():
        if pathway1 not in filtered_enrichr_df['Term'].values:
            continue
        for pathway2, genes2 in pathway_gene_map.items():
            if pathway1 != pathway2 and pathway2 in filtered_enrichr_df['Term'].values:
                shared_genes = genes1.intersection(genes2)
                if shared_genes:  # Add edge if there are shared genes
                    G.add_edge(pathway1, pathway2, weight=len(shared_genes))
                    shared_gene_labels[(pathway1, pathway2)] = '\n'.join(shared_genes)  # Insert newline

        # Non-shared genes for the pathway
        non_shared_genes = genes1 - set.union(*[genes2 for pathway2, genes2 in pathway_gene_map.items() if pathway2 != pathway1])
        #non_shared_gene_labels[pathway1] = '\n'.join(non_shared_genes)
        # Limit non-shared genes to the first 5 (or fewer if there are less than 5)
        non_shared_gene_labels[pathway1] = '\n'.join(list(non_shared_genes)[:3])  # Limit non-shared genes to first 5
        

    # Convert NetworkX graph to iGraph
    ig_graph = ig.Graph.TupleList(G.edges(data=False), directed=False)

    # Run Leiden clustering
    partition = ig_graph.community_leiden(resolution=0.7)  # Experiment with lower values for flexibility

    # Assign cluster labels to nodes (pathways)
    term_cluster_map = {ig_graph.vs[idx]['name']: cluster for idx, cluster in enumerate(partition.membership)}

    # Map clusters to filtered_enrichr_df DataFrame
    filtered_enrichr_df['Cluster'] = filtered_enrichr_df['Term'].map(term_cluster_map)

    # Dynamically calculate the figure size
    nconfpath = len(filtered_enrichr_df)
    print("Final number of confident pathways : " + str(nconfpath))

    xsize = ceil(nconfpath * 1.20)
    ysize = ceil(nconfpath * 1.00)
    xsize = max(xsize, 30)
    ysize = max(ysize, 25)
    #xsize = 30
    #ysize = 25
    print("Canvas size " + str(xsize) + " " + str(ysize))
    # Step 4: Visualize Graph with Improved Scale
    plt.figure(figsize=(xsize, ysize))  # Larger figure size for better visualization

    k0 = 1.0
    kf = 5.0
    kval = round(k0 + kf * (xsize - ysize) / xsize, 2)
    print("Final number k (spacing) : " + str(kval))

    # Adjust layout spacing with spring_layout
    # pos = nx.spring_layout(G, seed=42, k=kval, iterations=100)  # Increase 'k' for better spacing
    pos = nx.kamada_kawai_layout(G)  # Increase 'k' for better spacing

    # Map node size to gene count (increase size proportionally)
    min_gene_count = min(gene_counts.values())
    max_gene_count = max(gene_counts.values())

    # Dynamic scaling using min-max normalization with an adjustable range
    size_min = 1000  # Minimum node size
    size_max = 10000  # Maximum node size

    # Apply scaling
    if min_gene_count == max_gene_count:
        node_size = [size_max] * len(G.nodes())  # Use maximum size if counts are uniform
    else:
        node_size = [
            size_min + ((gene_counts[node] - min_gene_count) / (max_gene_count - min_gene_count)) * (size_max - size_min)
            for node in G.nodes()
        ]

    # Map node colors to clusters (color by cluster index using a colormap)
    if max(term_cluster_map.values()) == 0:
        print("All pathways belong to the same cluster. Assigning a default color.")
        color_map = ['red' for _ in term_cluster_map.values()]
    else:
        color_map = [plt.cm.hsv(cluster / max(term_cluster_map.values())) for cluster in term_cluster_map.values()]
        
    # Function to split text into groups of two words separated by '\n'
    def split_pathway_name(name):
        words = name.split()
        return '\n'.join([' '.join(words[i:i+2]) for i in range(0, len(words), 2)])

    # Adjust node labels to show each term with breaks every two words and number of genes
    labels = {node: f"{split_pathway_name(node)}\n({gene_counts.get(node, 0)} genes)\n{non_shared_gene_labels.get(node, '')}" for node in G.nodes()}
    
    npt_font = max(16, min(20, ceil(14 * kval)))  # Cap font size between 16 and 20
    print("Final npt_font (pathways) : " + str(npt_font))

    # Draw the graph
    nx.draw(
        G, pos,
        with_labels=True,
        labels=labels,  # Use updated labels with gene count and non-shared genes
        node_color=color_map,  # Color nodes by cluster with distinct colors
        node_size=node_size,
        font_size=npt_font,
        edge_color='gray',
        cmap=plt.cm.Set3
    )

    # Create a ScalarMappable object for colorbar reference
    sm = plt.cm.ScalarMappable(cmap=plt.cm.hsv, norm=Normalize(vmin=0, vmax=max(term_cluster_map.values())))
    sm.set_array([])  # Empty array for colorbar reference

    # Add a thinner colorbar using the 'fraction' and 'pad' arguments
    plt.colorbar(sm, ax=plt.gca(), label='Cluster Index', fraction=0.02, pad=0.02)

    if len(G.edges()) > 50:
        print("Too many edges. Skipping shared gene labels.")
    else:
        nx.draw_networkx_edge_labels(
            G, pos,
            edge_labels=shared_gene_labels,
            font_size=12  # Adjust font size for shared genes
        )

    plt.title('Pathway Analysis (P-value < 0.05) - Gene Overlap Graph', fontsize=30)

    # Save the plot to the specified output path
    plt.savefig(output_path, format='png', bbox_inches='tight')  # Save the plot as a PNG file
    print(f"Graph saved to {output_path}")

    plt.close()  # Close the plot to free up memory


In [49]:
import os
import pandas as pd

def open_matlab_outputfile(fname):
    """Open a file and filter genes based on the type (DV or DE)."""
    df = pd.read_excel(fname)

    # Get the filename from the full path
    filename = os.path.basename(fname)

    if "dv" in filename.lower():
        # Open DV summary spreadsheet
        print(f"Processing DV file: {filename}")
        # Keep important and confident genes
        colname = "DiffDist" # dist_diff
        idx = (df["pval"] < 0.05) & (df[colname] > 0)
        df = df[idx]
        gene_list = df["gene"]
        print(f"Number of confident genes (DV): {len(gene_list)}")
        return df, gene_list, "dv"

    else:
        # Open DE summary spreadsheet
        print(f"Processing DE file: {filename}")
        idx = (df['p_val_adj'] < 0.05) & (df['abs_log2FC'] >= 0.7)
        df = df[idx]
        gene_list = df['gene']
        print(f"Number of confident genes (DE): {len(gene_list)}")
        return df, gene_list, "de"

def open_gl(fname):
    df = pd.read_csv(fname, header=None)
    gl = df[0].to_list()
    return gl

In [72]:
fname = r"C:\Users\ssromerogon\Documents\vscode_working_dir\QUBO_Feature_Selection\qubo_fs_matlab\GSE134839_anticancer_drug_resistance_system\miscelaneous\qubo_features.txt"
gl = open_gl(fname)
out_name = r"qubo_enrichr_clustered.png"
plot_enrichr_pathway_analysis(gl, gene_set='GO_Biological_Process_2023', organism='human', output_path=out_name)

Number of confident pathways (start) : 38
Number of confident pathways (of interest): 38
Pathways with >= 3 genes found: 25
Final number of confident pathways : 25
Canvas size 30 25
Final number k (spacing) : 1.83
Final npt_font (pathways) : 20
Too many edges. Skipping shared gene labels.


C:\Users\ssromerogon\AppData\Local\Temp\ipykernel_36176\3752104744.py:101: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_enrichr_df = filtered_enrichr_df[idx]
c:\Local_install\miniconda3\envs\gseapy_env\Lib\site-packages\networkx\drawing\nx_pylab.py:457: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  node_collection = ax.scatter(


Graph saved to qubo_enrichr_clustered.png


In [73]:
fname = r"C:\Users\ssromerogon\Documents\vscode_working_dir\QUBO_Feature_Selection\qubo_fs_matlab\GSE134839_anticancer_drug_resistance_system\miscelaneous\lasso_features.txt"
gl = open_gl(fname)
out_name = r"lasso_enrichr_clustered.png"
plot_enrichr_pathway_analysis(gl, gene_set='GO_Biological_Process_2023', organism='human', output_path=out_name)

Number of confident pathways (start) : 76
Number of confident pathways (of interest): 76
Pathways with >= 3 genes found: 37
Final number of confident pathways : 37
Canvas size 45 37
Final number k (spacing) : 1.89
Final npt_font (pathways) : 20
Too many edges. Skipping shared gene labels.


C:\Users\ssromerogon\AppData\Local\Temp\ipykernel_36176\3752104744.py:101: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_enrichr_df = filtered_enrichr_df[idx]
c:\Local_install\miniconda3\envs\gseapy_env\Lib\site-packages\networkx\drawing\nx_pylab.py:457: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  node_collection = ax.scatter(


Graph saved to lasso_enrichr_clustered.png


In [74]:
fname = r"C:\Users\ssromerogon\Documents\vscode_working_dir\QUBO_Feature_Selection\qubo_fs_matlab\GSE134839_anticancer_drug_resistance_system\miscelaneous\fittree_features.txt"
gl = open_gl(fname)
out_name = r"fittree_enrichr_clustered.png"
plot_enrichr_pathway_analysis(gl, gene_set='GO_Biological_Process_2023', organism='human', output_path=out_name)

Number of confident pathways (start) : 39
Number of confident pathways (of interest): 39
Pathways with >= 3 genes found: 26
Final number of confident pathways : 26
Canvas size 32 26
Final number k (spacing) : 1.94
Final npt_font (pathways) : 20
Too many edges. Skipping shared gene labels.


C:\Users\ssromerogon\AppData\Local\Temp\ipykernel_36176\3752104744.py:101: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_enrichr_df = filtered_enrichr_df[idx]
c:\Local_install\miniconda3\envs\gseapy_env\Lib\site-packages\networkx\drawing\nx_pylab.py:457: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  node_collection = ax.scatter(


Graph saved to fittree_enrichr_clustered.png


In [ ]:
import pandas as pd
import networkx as nx
import igraph as ig
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import re
from math import ceil
from matplotlib.colors import Normalize
import gseapy as gp
 
def enrichr_analysis(gene_list, gene_set='GO_Biological_Process_2023', organism='mouse'):
    """Perform Enrichr analysis and return filtered results."""
    try:
        enrichr_results = gp.enrichr(
            gene_list=gene_list,
            gene_sets=gene_set,
            organism=organism
        )
        enrichr_df = enrichr_results.results
    except Exception as e:
        print(f"Enrichr analysis failed: {e}")
        return None
   
    # Adjust p-values and filter significant terms
    pval_threshold = 0.05
    enrichr_df = enrichr_df[enrichr_df['Adjusted P-value'] <= pval_threshold]
   
    return enrichr_df[['Term', 'Adjusted P-value', 'Genes']]
 
def cluster_enriched_terms_df(enrichr_df, output_path='compared_pathways_clusters.png'):
    # Step 1: Filter and clean up pathways
    print(f"Number of pathways before filtering: {len(enrichr_df)}")
   
    filtered_enrichr_df = enrichr_df
 
    # Print the filtered DataFrame
    nconfpath = len(filtered_enrichr_df)
    if nconfpath == 0:  # Check if the filtered DataFrame is empty
        print("No pathways match the specified keywords. Exiting analysis.")
        return
 
 
    # Step 2: Proceed with the graph creation using the filtered DataFrame
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform(filtered_enrichr_df['Term'])
 
    # Step 3: Build Graph from Shared Genes
    G = nx.Graph()
    gene_counts = {}
    pathway_gene_map = {}
 
    for _, row in filtered_enrichr_df.iterrows():
        genes = set(row['Genes'].split(';'))  # Convert to set for intersection checks
        pathway_gene_map[row['Term']] = genes
        gene_counts[row['Term']] = len(genes)  # Count genes per term
 
    #filtered_enrichr_df['num_genes'] = filtered_enrichr_df['Term'].map(gene_counts)
    filtered_enrichr_df = filtered_enrichr_df.copy()
    filtered_enrichr_df.loc[:, 'num_genes'] = filtered_enrichr_df['Term'].map(gene_counts)
 
    # Filter pathways that have >= 3 genes, with fallback to >= 2 and >= 1
    #for threshold in [3, 2, 1]:
    for threshold in [3, 2]:
        idx = filtered_enrichr_df['num_genes'] >= threshold
        if idx.sum() > 3:
            print(f"Pathways with >= {threshold} genes found: {idx.sum()}")
            filtered_enrichr_df = filtered_enrichr_df[idx]
            break
    else:
        # If no pathways meet the minimum gene count, exit the function
        print("No pathways with at least 2 gene found. Skipping further analysis.")
        return
 
    filtered_enrichr_df = filtered_enrichr_df[idx]
 
    # Recalculate gene counts for the filtered pathways
    gene_counts = {term: gene_counts[term] for term in filtered_enrichr_df['Term']}
 
    # Add edges only for shared genes
    shared_gene_labels = {}
    non_shared_gene_labels = {}
    for pathway1, genes1 in pathway_gene_map.items():
        if pathway1 not in filtered_enrichr_df['Term'].values:
            continue
        for pathway2, genes2 in pathway_gene_map.items():
            if pathway1 != pathway2 and pathway2 in filtered_enrichr_df['Term'].values:
                shared_genes = genes1.intersection(genes2)
                if shared_genes:  # Add edge if there are shared genes
                    G.add_edge(pathway1, pathway2, weight=len(shared_genes))
                    shared_gene_labels[(pathway1, pathway2)] = '\n'.join(shared_genes)  # Insert newline
 
        # Non-shared genes for the pathway
        non_shared_genes = genes1 - set.union(*[genes2 for pathway2, genes2 in pathway_gene_map.items() if pathway2 != pathway1])
        #non_shared_gene_labels[pathway1] = '\n'.join(non_shared_genes)
        # Limit non-shared genes to the first 5 (or fewer if there are less than 5)
        non_shared_gene_labels[pathway1] = '\n'.join(list(non_shared_genes)[:3])  # Limit non-shared genes to first 5
       
 
    # Convert NetworkX graph to iGraph
    ig_graph = ig.Graph.TupleList(G.edges(data=False), directed=False)
 
    # Run Leiden clustering
    partition = ig_graph.community_leiden(resolution=0.7)  # Experiment with lower values for flexibility
 
    # Assign cluster labels to nodes (pathways)
    term_cluster_map = {ig_graph.vs[idx]['name']: cluster for idx, cluster in enumerate(partition.membership)}
 
    # Map clusters to filtered_enrichr_df DataFrame
    filtered_enrichr_df['Cluster'] = filtered_enrichr_df['Term'].map(term_cluster_map)
 
    # Dynamically calculate the figure size
    nconfpath = len(filtered_enrichr_df)
    print("Final number of confident pathways : " + str(nconfpath))
 
    xsize = ceil(nconfpath * 1.20)
    ysize = ceil(nconfpath * 1.00)
    xsize = max(xsize, 30)
    ysize = max(ysize, 25)
    #xsize = 30
    #ysize = 25
    print("Canvas size " + str(xsize) + " " + str(ysize))
    # Step 4: Visualize Graph with Improved Scale
    plt.figure(figsize=(xsize, ysize))  # Larger figure size for better visualization
 
    k0 = 1.0
    kf = 5.0
    kval = round(k0 + kf * (xsize - ysize) / xsize, 2)
    print("Final number k (spacing) : " + str(kval))
 
    # Adjust layout spacing with spring_layout
    pos = nx.spring_layout(G, seed=42, k=kval, iterations=100)  # Increase 'k' for better spacing
 
    # Map node size to gene count (increase size proportionally)
    min_gene_count = min(gene_counts.values())
    max_gene_count = max(gene_counts.values())
 
    # Dynamic scaling using min-max normalization with an adjustable range
    size_min = 1000  # Minimum node size
    size_max = 10000  # Maximum node size
 
    # Apply scaling
    if min_gene_count == max_gene_count:
        node_size = [size_max] * len(G.nodes())  # Use maximum size if counts are uniform
    else:
        node_size = [
            size_min + ((gene_counts[node] - min_gene_count) / (max_gene_count - min_gene_count)) * (size_max - size_min)
            for node in G.nodes()
        ]
 
    # Map node colors to clusters (color by cluster index using a colormap)
    if max(term_cluster_map.values()) == 0:
        print("All pathways belong to the same cluster. Assigning a default color.")
        color_map = ['red' for _ in term_cluster_map.values()]
    else:
        color_map = [plt.cm.hsv(cluster / max(term_cluster_map.values())) for cluster in term_cluster_map.values()]
       
    # Function to split text into groups of two words separated by '\n'
    def split_pathway_name(name):
        words = name.split()
        return '\n'.join([' '.join(words[i:i+2]) for i in range(0, len(words), 2)])
 
    # Adjust node labels to show each term with breaks every two words and number of genes
    labels = {node: f"{split_pathway_name(node)}\n({gene_counts.get(node, 0)} genes)\n{non_shared_gene_labels.get(node, '')}" for node in G.nodes()}
   
    npt_font = max(16, min(20, ceil(14 * kval)))  # Cap font size between 16 and 20
    print("Final npt_font (pathways) : " + str(npt_font))
 
    # Draw the graph
    nx.draw(
        G, pos,
        with_labels=True,
        labels=labels,  # Use updated labels with gene count and non-shared genes
        node_color=color_map,  # Color nodes by cluster with distinct colors
        node_size=node_size,
        font_size=npt_font,
        edge_color='gray',
        cmap=plt.cm.Set3
    )
 
    # Create a ScalarMappable object for colorbar reference
    sm = plt.cm.ScalarMappable(cmap=plt.cm.hsv, norm=Normalize(vmin=0, vmax=max(term_cluster_map.values())))
    sm.set_array([])  # Empty array for colorbar reference
 
    # Add a thinner colorbar using the 'fraction' and 'pad' arguments
    plt.colorbar(sm, ax=plt.gca(), label='Cluster Index', fraction=0.02, pad=0.02)
 
    if len(G.edges()) > 50:
        print("Too many edges. Skipping shared gene labels.")
    else:
        nx.draw_networkx_edge_labels(
            G, pos,
            edge_labels=shared_gene_labels,
            font_size=12  # Adjust font size for shared genes
        )
 
    plt.title('Pathway Analysis (P-value < 0.05) - Gene Overlap Graph', fontsize=30)
 
    # Save the plot to the specified output path
    plt.savefig(output_path, format='png', bbox_inches='tight')  # Save the plot as a PNG file
    print(f"Graph saved to {output_path}")
 
    plt.close()  # Close the plot to free up memory
 
 
def compare_enriched_terms(gene_list1, gene_list2, gene_set='GO_Biological_Process_2023', organism='mouse', output_path='compared_pathways_clusters.png'):
    """Perform Enrichr analysis for two gene lists, remove common terms, and plot unique terms for gene_list1."""
   
    # Run Enrichr for both gene lists
    df1 = enrichr_analysis(gene_list1, gene_set, organism)
    df2 = enrichr_analysis(gene_list2, gene_set, organism)
   
    if df1 is None or df2 is None:
        print("Enrichr analysis failed for one or both gene lists.")
        return
   
    # Find common terms
    common_terms = set(df1['Term']).intersection(set(df2['Term']))
   
    # Filter df1 to remove common terms
    df1_filtered = df1[~df1['Term'].isin(common_terms)]
   
    print(f"Removed {len(common_terms)} common pathways.")
    print(f"Remaining pathways for Gene List 1: {len(df1_filtered)}")
   
    if df1_filtered.empty:
        print("No unique pathways left for plotting.")
        return
   
    # Proceed with pathway clustering and plotting
    cluster_enriched_terms_df(df1_filtered, output_path=output_path)

In [89]:
fname = r"C:\Users\ssromerogon\Documents\vscode_working_dir\QUBO_Feature_Selection\qubo_fs_matlab\GSE134839_anticancer_drug_resistance_system\miscelaneous\qubo_features.txt"
gl1 = open_gl(fname)
fname = r"C:\Users\ssromerogon\Documents\vscode_working_dir\QUBO_Feature_Selection\qubo_fs_matlab\GSE134839_anticancer_drug_resistance_system\miscelaneous\lasso_features.txt"
gl2 = open_gl(fname)

compare_enriched_terms(gl1, gl2, gene_set='GO_Biological_Process_2023', organism='human', output_path='qubo_only_vs_lasso_clustered_pathways.png' )

Removed 16 common pathways.
Remaining pathways for Gene List 1: 22
Number of pathways: 22
Canvas size: 30 25
No clusters were found. Using a default color map
Graph saved to qubo_only_vs_lasso_clustered_pathways.png


In [71]:
fname = r"C:\Users\ssromerogon\Documents\vscode_working_dir\QUBO_Feature_Selection\qubo_fs_matlab\GSE134839_anticancer_drug_resistance_system\miscelaneous\qubo_features.txt"
gl1 = open_gl(fname)
fname = r"C:\Users\ssromerogon\Documents\vscode_working_dir\QUBO_Feature_Selection\qubo_fs_matlab\GSE134839_anticancer_drug_resistance_system\miscelaneous\fittree_features.txt"
gl2 = open_gl(fname)

compare_enriched_terms(gl1, gl2, gene_set='GO_Biological_Process_2023', organism='human', output_path='qubo_only_vs_fittree_clustered_pathways.png' )

Removed 24 common pathways.
Remaining pathways for Gene List 1: 14
Number of pathways before filtering: 14
Pathways with >= 3 genes found: 12
Final number of confident pathways : 12
Canvas size 30 25
Final number k (spacing) : 1.83
Final npt_font (pathways) : 20


C:\Users\ssromerogon\AppData\Local\Temp\ipykernel_36176\3229064636.py:74: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_enrichr_df = filtered_enrichr_df[idx]
c:\Local_install\miniconda3\envs\gseapy_env\Lib\site-packages\networkx\drawing\nx_pylab.py:457: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  node_collection = ax.scatter(


Graph saved to qubo_only_vs_fittree_clustered_pathways.png
